# Seeded Clustering — Calibration on the ELSST train split

Seeds are drawn uniformly at random from all terms (no gold information). Every setting (number of seeds, similarity
threshold) is repeated with `n_trials` seed draws and the setting with the highest mean F1 is kept.

In [1]:
# project paths and the device come from scripts/config.py (works locally and on Colab)
import sys
import os
import gc
from pathlib import Path
import importlib

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'analysis').exists():
            return candidate
    raise RuntimeError("Could not locate project root containing scripts/ and analysis/")

sys.path.insert(0, str(_find_project_root(Path.cwd().resolve())))

import numpy as np
import pandas as pd
import torch
from scripts.model_utils_seed import run_seed_clustering_optimization
from scripts import model_utils_shared, config
import warnings

# Reload config to get latest changes
importlib.reload(config)

# Silence warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Reproducibility + deterministic output paths
SEED = model_utils_shared.setup_reproducibility(config.SEED)

PROJECT_ROOT = Path(config.PROJECT_ROOT)
ELSST_CALLIBRATIONS_DIR = Path(config.CALIBRATION_DIR)
ELSST_PLOTS_DIR = Path(config.CALIBRATION_PLOTS_DIR)
ELSST_CALLIBRATIONS_DIR.mkdir(parents=True, exist_ok=True)
ELSST_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Calibration results: {ELSST_CALLIBRATIONS_DIR}")
print(f"Device: {config.DEVICE}")
print(f"Using seed: {SEED}")


/home/rass/Desktop/projects/Social_Science_Construct_Harmonization/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /home/rass/Desktop/projects/Social_Science_Construct_Harmonization
Calibration results: /home/rass/Desktop/projects/Social_Science_Construct_Harmonization/analysis/results/elsst/callibrations_results
Device: cpu
Using seed: 42


In [2]:
# Calibration data: ELSST train split only (Parquet files with integer term ids).
# Negatives use all labels of both concepts. idx1 / idx2 point into `terms`.
data = model_utils_shared.load_pair_data(**config.DATASETS["elsst_train"])
terms = data["terms"]
idx1, idx2, labels = data["idx1"], data["idx2"], data["label"]
print(f"Total pairs: {len(labels):,} | positives: {int(labels.sum()):,} | unique terms: {len(terms):,}")


Loading pair data from /home/rass/Desktop/projects/Social_Science_Construct_Harmonization/datasets/processed_datasets/elsst (prefix 'train_')...
  terms: 4,270 | positive pairs: 3,587 | negative pairs: 9,098,662 | broader/narrower pairs: 12,066
Total pairs: 9,102,249 | positives: 3,587 | unique terms: 4,270


In [3]:
models_to_test = config.MODELS
random_seed = config.SEED
n_initial_seeds = config.SEED_CLUSTER_PARAMS['n_initial_seeds_list']
n_random_trials = config.SEED_CLUSTER_PARAMS['n_trials']
similarity_thresholds = config.SEED_CLUSTER_PARAMS['thresholds']
all_seed_clustering_results = []

In [4]:
# Seeds are drawn at random from all terms (no gold concept ids); embeddings are
# computed once per model and passed to the grid search
for model_name, model_info in models_to_test.items():
    print(f"\nLoading model: {model_name} | model type: {model_info['type']}")
    model = model_utils_shared.load_model(model_name=model_name, model_type=model_info['type'], device=config.DEVICE)
    if model is None:
        continue
    emb = model_utils_shared.build_embeddings(model, terms, batch_size=config.BATCH_SIZE)
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    df_results, grouped, best_n_initial, best_threshold, best_f1_mean = run_seed_clustering_optimization(
        emb["norm_t"], idx1, idx2, labels,
        n_initial_seeds=n_initial_seeds,
        similarity_threshold=similarity_thresholds,
        random_state=random_seed,
        num_random_trials=n_random_trials
    )
    del emb
    gc.collect()

    # Store results
    all_seed_clustering_results.append({
        'model_name': model_name,
        'df_results': df_results,
        'grouped': grouped,
        'best_n_initial': best_n_initial,
        'best_threshold': best_threshold,
        'best_f1_mean': best_f1_mean
    })

    print("\n" + "="*80)
    print(f"MODEL: {model_name}")
    print(f"BEST CONFIGURATION:")
    print(f"  Seeds: {int(best_n_initial)} | Threshold: {best_threshold:.2f} | Mean F1: {best_f1_mean:.4f}")
    print("="*80)


Loading model: all-mpnet-base-v2 | model type: sentence_transformer
Loading Model (all-mpnet-base-v2) on cpu...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10326.96it/s]


MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Encoding 4270 unique terms...


Batches:   0%|          | 0/267 [00:00<?, ?it/s]

... (3,248 outputs left out)

Seeds:  500|Threshold:   0.95|Trial:  7|F1:  0.0308|Precision:  0.1406|Recall:  0.0173|Clusters: 4155


Seeds:  500|Threshold:   0.95|Trial:  8|F1:  0.0314|Precision:  0.1708|Recall:  0.0173|Clusters: 4158


Seeds:  500|Threshold:   0.95|Trial:  9|F1:  0.0334|Precision:  0.2117|Recall:  0.0181|Clusters: 4152


Seeds:  500|Threshold:   0.95|Trial: 10|F1:  0.0334|Precision:  0.3351|Recall:  0.0176|Clusters: 4158

MODEL: bert-base-uncased
BEST CONFIGURATION:
  Seeds: 500 | Threshold: 0.90 | Mean F1: 0.0686


In [5]:
# Summary: Best configurations for all models
print("\n" + "="*100)
print("SUMMARY: BEST CONFIGURATIONS FOR ALL MODELS")
print("="*100)

summary_list = []
for result in all_seed_clustering_results:
    model_name = result['model_name']
    best_n_initial = result['best_n_initial']
    best_threshold = result['best_threshold']
    best_f1_mean = result['best_f1_mean']
    
    summary_list.append({
        'Model': model_name,
        'Best Seeds': int(best_n_initial),
        'Best Threshold': f"{best_threshold:.2f}",
        'Mean F1': f"{best_f1_mean:.4f}"
    })

summary_df = pd.DataFrame(summary_list)
print(summary_df.to_string(index=False))
print("="*100)


SUMMARY: BEST CONFIGURATIONS FOR ALL MODELS
                           Model  Best Seeds Best Threshold Mean F1
               all-mpnet-base-v2         250           0.75  0.4215
        dwulff/mpnet-personality          10           0.75  0.3663
allenai/scibert_scivocab_uncased         500           0.90  0.1205
               bert-base-uncased         500           0.90  0.0686


In [6]:
# Combine individual results from all models and save to CSV
individual_results_list = []
for result in all_seed_clustering_results:
    model_name = result['model_name']
    df_results = result['df_results'].copy()
    df_results['model_name'] = model_name
    individual_results_list.append(df_results)

all_individual_results_df = pd.concat(individual_results_list, ignore_index=True)

# Save to CSV

csv_path = ELSST_CALLIBRATIONS_DIR / 'seed_cluster_results.csv'
all_individual_results_df.to_csv(csv_path, index=False)
print(f"\nSaved individual results to: {csv_path}")
print(f"Total rows: {len(all_individual_results_df)}")
print(all_individual_results_df.head())

# copy these lines into scripts/config.py
print("\nPaste into scripts/config.py:\n")
print("BEST_SEEDED_PARAMS = {")
for result in all_seed_clustering_results:
    print(f'    "{result["model_name"]}": {{"n_initial_seeds": {int(result["best_n_initial"])}, '
          f'"threshold": {result["best_threshold"]:.2f}}},')
print("}")


Saved individual results to: /home/rass/Desktop/projects/Social_Science_Construct_Harmonization/analysis/results/elsst/callibrations_results/seed_cluster_results.csv
Total rows: 4320
   threshold  trial  precision    recall        f1   pos_acc   neg_acc  \
0        0.1      1   0.001762  0.686646  0.003516  0.686646  0.846669   
1        0.1      2   0.002466  0.741288  0.004916  0.741288  0.881804   
2        0.1      3   0.001796  0.706161  0.003583  0.706161  0.845289   
3        0.1      4   0.002172  0.765263  0.004332  0.765263  0.861401   
4        0.1      5   0.001832  0.677168  0.003654  0.677168  0.854526   

   num_initial_seeds  n_clusters         model_name  
0                 10          10  all-mpnet-base-v2  
1                 10          10  all-mpnet-base-v2  
2                 10          10  all-mpnet-base-v2  
3                 10          10  all-mpnet-base-v2  
4                 10          11  all-mpnet-base-v2  

Paste into scripts/config.py:

BEST_SEEDED_PAR